# Student Performance Analysis and Prediction using AI

**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI Academic Internship Program  
**Student:** Manish  
**Dataset:** UCI Student Performance Dataset

## Objective
Analyze student demographic, social, family, study and academic factors and build a machine-learning model to predict final student performance.

> **Important:** G1 and G2 are earlier-period grades and are strongly related to G3 (the final grade). The notebook therefore evaluates a model using all available predictors and also demonstrates a more realistic "early prediction" version that excludes G1 and G2.


In [ ]:
# Install in a fresh environment if required:
# !pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


In [ ]:
# Load the public UCI dataset
student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features.copy()
y = student_performance.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.head())


## 1. Data Understanding

The UCI dataset contains student achievement information from two Portuguese schools. It includes demographic, social, school-related and grade information. The final grade **G3** is the target used for prediction.

In [ ]:
# Combine features and target
df = X.copy()
if isinstance(y, pd.DataFrame):
    target_col = y.columns[0]
    df[target_col] = y[target_col].values
else:
    target_col = "G3"
    df[target_col] = np.asarray(y)

print("Dataset shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDescriptive statistics:")
display(df.describe(include="all").T.head(35))


In [ ]:
# Distribution of final grades
plt.figure(figsize=(9, 5))
sns.histplot(df[target_col], bins=21, kde=True)
plt.title("Distribution of Final Grade (G3)")
plt.xlabel("Final Grade")
plt.ylabel("Number of Students")
plt.show()

# Average final grade by study time
if "studytime" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x="studytime", y=target_col, errorbar=None)
    plt.title("Average Final Grade by Study Time")
    plt.xlabel("Study Time Category")
    plt.ylabel("Average G3")
    plt.show()

# Absences vs final grade
if "absences" in df.columns:
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=df, x="absences", y=target_col)
    plt.title("Absences vs Final Grade")
    plt.xlabel("Absences")
    plt.ylabel("Final Grade")
    plt.show()


In [ ]:
# Correlation analysis for numeric variables
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 9))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

if target_col in numeric_df.columns:
    corr = numeric_df.corr()[target_col].sort_values(ascending=False)
    print("Correlation with final grade:")
    display(corr.to_frame("correlation"))


## 2. Data Preparation

Categorical variables are one-hot encoded and numerical variables are standardized. Missing-value handling is included in the pipeline so the workflow remains reproducible.

In [ ]:
# Build preprocessing pipeline
X_model = df.drop(columns=[target_col])
y_model = df[target_col]

categorical_cols = X_model.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X_model.select_dtypes(include=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])


## 3. AI Model — Random Forest Regression

Random Forest Regression is used because the target **G3** is numeric (0–20). The model combines multiple decision trees and can capture non-linear relationships between student characteristics and final performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.20, random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        max_depth=None,
        min_samples_leaf=2
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")


In [ ]:
# Actual vs predicted grades
results = pd.DataFrame({
    "Actual_G3": y_test.values,
    "Predicted_G3": np.round(pred, 2)
}).reset_index(drop=True)

display(results.head(15))

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=pred)
plt.plot([0, 20], [0, 20], linestyle="--")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title("Actual vs Predicted Final Grade")
plt.show()


## 4. Early-Prediction Model

For a more practical use case, G1 and G2 are removed. This avoids using earlier grades that are very close to the final outcome and tests whether demographic, social, study and attendance variables alone can help predict final performance.

In [ ]:
early_drop = [c for c in ["G1", "G2"] if c in X_model.columns]
X_early = X_model.drop(columns=early_drop)

cat_early = X_early.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_early = X_early.select_dtypes(include=np.number).columns.tolist()

preprocessor_early = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_early),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_early)
])

Xtr, Xte, ytr, yte = train_test_split(
    X_early, y_model, test_size=0.20, random_state=42
)

early_model = Pipeline([
    ("preprocessor", preprocessor_early),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2
    ))
])

early_model.fit(Xtr, ytr)
early_pred = early_model.predict(Xte)

early_mae = mean_absolute_error(yte, early_pred)
early_rmse = np.sqrt(mean_squared_error(yte, early_pred))
early_r2 = r2_score(yte, early_pred)

print("Early-prediction model (without G1/G2)")
print(f"MAE  : {early_mae:.3f}")
print(f"RMSE : {early_rmse:.3f}")
print(f"R²   : {early_r2:.3f}")


## 5. Key Findings

The notebook should be interpreted using the actual outputs generated above. Important analytical questions include:

- How strongly do G1 and G2 relate to G3?
- Does study time show a visible relationship with final grade?
- How are absences associated with final performance?
- Which social, family and academic variables appear useful for prediction?
- How much does predictive performance change after removing G1 and G2?

### Limitations
- The dataset represents two Portuguese schools and should not be treated as a universal sample of all students.
- Correlation does not prove causation.
- The model is intended for educational analytics, not for making high-stakes decisions about individual students.
- Results depend on the train/test split and model configuration.


## 6. Conclusion

This project demonstrates a complete data analytics and AI workflow: public-data acquisition, cleaning, exploratory analysis, visualization, preprocessing, machine-learning regression and evaluation. The early-prediction experiment also demonstrates why feature selection matters when building a realistic predictive system.

### Dataset Source
UCI Machine Learning Repository — Student Performance:
https://archive.ics.uci.edu/dataset/320/student+performance

**Dataset citation:** Cortez, P. (2008). Student Performance. UCI Machine Learning Repository. DOI: 10.24432/C5TG7T.
